# 1. 패키지 설치

In [1]:
# 패키지 설치
%pip install -U python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters docx2txt langchain-chroma langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Knowledge Base 구성을 위한 데이터 생성

- `RecursiveCharacterTextSplitter`를 활용한 데이터 chunking
    - split 된 chunk를 LLM에 전달하면 토큰 절약 (비용·답변시간 감소)
- `chunk_size`: chunk 최대 크기 / `chunk_overlap`: 앞뒤 chunk가 겹치는 정도

In [3]:
# docx 문서 Load & Split
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
# 환경변수 Load + Upstage 임베딩 설정 (model은 필수 인자)
# solar-embedding-1-large: 내부에서 -query/-passage 자동 부착
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

In [5]:
# Vector DB(Chroma)
from langchain_chroma import Chroma

# 데이터 처음 저장
database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory='./chroma')

# 이미 저장된 데이터 사용
# database = Chroma(collection_name='chroma-tax', persist_directory='./chroma', embedding_function=embedding)

# 3. 답변 생성을 위한 Retrieval

- `Chroma`에 저장한 데이터를 유사도 검색(`similarity_search()`)으로 가져옴

In [7]:
# Retrieval: Chroma DB에서 유사도 기준 Top 3 검색 (similarity_search)
# (참고: Retriever 객체는 database.as_retriever()로 별도 생성)
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'

retrieved_docs = database.similarity_search(query, k=3)

In [8]:
retrieved_docs

[Document(id='d8f9437b-4634-460a-8143-1a06eb1548b9', metadata={'source': './tax.docx'}, page_content='2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원\n\n2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원\n\n3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원\n\n④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>\n\n[본조신설 2014. 1. 1.]\n\n[종전 제59조의2는 제59조의5로 이동 <2014. 1. 1.>]\n\n\n\n제59조의3(연금계좌세액공제) ① 종합소득이 있는 거주자가 연금계좌에 납입한 금액 중 다음 각 호에 해당하는 금액을 제외한 금액(이하 “연금계좌 납입액”이라 한다)의 100분의 12[해당 과세기간에 종합소득과세표준을 계산할 때 합산하는 종합소득금액이 4천 500만원 이하(근로소득만 있는 경우에는 총급여액 5천 500만원 이하)인 거주자에 대해서는 100분의 15]에 해당하는 금액을 해당 과세기간의 종합소득산출세액에서 공제한다. 다만, 연금계좌 중 연금저축계좌에 납입한 금액이 연 600만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 하고, 연금저축계좌에 납입한 금액 중 600만원 이내의 금액과 퇴직연금계좌에 납입한 금액을 합한 금액이 연 900만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 한다. <개정 2014. 12. 23., 2015.

# 4. Augmentation을 위한 Prompt 활용

- 자주 쓰는 표준 RAG 프롬프트(`rlm/rag-prompt`)를 Hub에서 가져와 사용

In [9]:
# LangChain 기반 LLM 설정 (Upstage)
# ChatUpstage 기본 모델 = solar-mini (품질 원하면 model='solar-pro' 등 지정 가능)
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

In [10]:
# 프롬프트: 표준 RAG 프롬프트를 Hub에서 가져옴 (rlm/rag-prompt = Human 메시지, 변수 context·question)
# langchain.hub는 v1에서 제거됨, langchain_classic.hub.pull도 deprecated(2.0.0 제거 예정) → langsmith로 pull
from langsmith import Client

client = Client()
prompt = client.pull_prompt('rlm/rag-prompt', dangerously_pull_public_prompt=True)

In [11]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

# 5. 답변 생성

- `RetrievalQA`를 통해 LLM에 전달
    - `RetrievalQA`는 `create_retrieval_chain`으로 대체됨 (강의 3.2.2에서 변경 과정 학습)

In [12]:
# 답변 생성: RetrievalQA로 (검색 결과 + 프롬프트)를 LLM에 전달
# ⚠️ RetrievalQA는 이미 deprecated(0.1.17~), langchain-classic 2.0.0에서 제거 예정 → 학습용으로만 사용
# 대체: create_retrieval_chain (강의 3.2.2) / 최신 권장: create_agent
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={'prompt': prompt}
)

In [ ]:
# Chain을 통한 LLM 호출
ai_msg = qa_chain.invoke({'query': query})

In [ ]:
# 출력 확인
ai_msg